In [14]:
import pandas as pd
import numpy as np

df_user = pd.read_excel("../RetrievalModel/Data/top30k_user.xlsx")

# fill missing rating / helpful votes
df_user['rating'] = df_user['rating'].fillna(df_user['rating'].mean())
df_user['helpful_vote'] = df_user['helpful_vote'].fillna(0)

# set base probabilities
p_click, p_cart, p_fav = 0.3, 0.2, 0.1 
bonus = (df_user['rating'] -1) / 4  # rating from 0-5; 4 actions: click, cart, favourite, purchase
df_user['p_click'] = (p_click + 0.3 * bonus).clip(0, 1)
df_user['p_cart'] = (p_cart + 0.2 * bonus).clip(0, 1)
df_user['p_fav'] = (p_fav + 0.1 * bonus).clip(0, 1)

# sample actions
def sample_action(row):
    if row['verified_purchase']:
        return 4 # purchase
    if np.random.rand() < row['p_cart']:
        return 2 # add to cart
    if np.random.rand() < row['p_fav']:
        return 3 # favorite
    if np.random.rand() < row['p_click']:
        return 1 # click
    return 0 # view

df_user['action_label'] = df_user.apply(sample_action, axis= 1)

# Inspect distribution
print(df_user['action_label'].value_counts(normalize=True))

df_user.to_excel("Data/top30k_user_labels.xlsx")

action_label
4    0.774490
2    0.082556
1    0.062675
0    0.054437
3    0.025841
Name: proportion, dtype: float64


In [17]:
df_user[df_user['verified_purchase']].shape

(58474, 15)

In [19]:
df_user[~ df_user['verified_purchase']].shape

(17026, 15)

In [20]:
58474 / (58474 + 17026)

0.7744900662251656